# Task 2.2: Reproduction of (ℓ₂)∞SVM(σ)

**Paper**: *A Simple Geometric Interpretation of SVM using Stochastic Adversaries* — Livni, Crammer, Globerson (AISTATS 2012)

## What we are reproducing

I am reproducing the core contribution: **Theorem 3.1**, which shows that RSVM2(σ) is equivalent to **(ℓ₂)∞SVM(σ)** — regularizing multiclass hinge loss with the **ℓ₂,∞ norm** (max of per-class weight norms) instead of the standard Frobenius norm.

**Evaluation metric**: Classification accuracy (equivalently error rate, as in Figure 1 of the paper).

The key insight of the paper is that robustness to stochastic adversarial perturbations in the input space leads to a natural regularization term. When the adversary's budget is measured in ℓ₂ norm, the resulting regularizer is the ℓ₂,∞ norm of the weight matrix (the maximum ℓ₂ norm across class-specific weight vectors), which differs from the standard Frobenius norm used in Crammer & Singer's multiclass SVM.

In [1]:
# ============================================================
# Hyperparameters and setup — all defined in one cell
# ============================================================
import numpy as np
np.random.seed(42)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import LinearSVC

# Hyperparameters
RANDOM_SEED = 42
TEST_SIZE = 0.2
N_ITERS = 2000          # Number of subgradient iterations
LR = 0.1                # Base learning rate
EVAL_EVERY = 50         # Evaluate objective every N iterations

print("Setup complete. Hyperparameters:")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Test size: {TEST_SIZE}")
print(f"  Subgradient iterations: {N_ITERS}")
print(f"  Base learning rate: {LR}")
print(f"  Evaluate every: {EVAL_EVERY} iterations")

Setup complete. Hyperparameters:
  Random seed: 42
  Test size: 0.2
  Subgradient iterations: 2000
  Base learning rate: 0.1
  Evaluate every: 50 iterations


## Data Loading

We load the same dataset and apply the same preprocessing as in Task 2.1: `sklearn.datasets.load_digits`, standardized to zero mean and unit variance, split 80/20 with `random_state=42`.

In [2]:
# Load and preprocess data (same as Task 2.1)
digits = load_digits()
X_raw, y = digits.data, digits.target

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

n_train, d = X_train.shape
L = len(np.unique(y))  # number of classes

print(f"Training: {n_train} samples, {d} features, {L} classes")
print(f"Test: {X_test.shape[0]} samples")

Training: 1437 samples, 64 features, 10 classes
Test: 360 samples


## 1. Multiclass Hinge Loss (Eq. 1)

From **Equation 1** of the paper, the multiclass hinge loss for a single example $(x, y)$ with weight matrix $W$ is:

$$\ell(x, y; W) = \max_{\bar{y}} \left[ w_{\bar{y}} \cdot x - w_y \cdot x + e(y, \bar{y}) \right]$$

where $e(y, \bar{y}) = \mathbb{1}[y \neq \bar{y}]$ (0-1 loss). This is the Crammer & Singer multiclass hinge loss. The overall loss is the average over all training examples.

In [3]:
def multiclass_hinge_loss(X, y, W, L):
    """
    Eq. 1: ℓ(x,y;W) = max_ȳ [wȳ·x − wy·x + e(y,ȳ)]
    
    Computes the average multiclass hinge loss over all samples.
    
    Parameters:
        X: (n, d) feature matrix
        y: (n,) label vector
        W: (L, d) weight matrix (one row per class)
        L: number of classes
    
    Returns:
        Scalar average hinge loss
    """
    n = X.shape[0]
    scores = X @ W.T                                        # (n, L) — score for each class
    margins = scores - scores[np.arange(n), y].reshape(-1, 1) + 1.0  # wȳ·x − wy·x + 1
    margins[np.arange(n), y] = 0.0                          # e(y,y) = 0, so true class margin = 0
    loss = np.maximum(margins, 0).max(axis=1)               # max over ȳ
    return np.mean(loss)

# Quick test with random weights
W_test = np.random.randn(L, d) * 0.01
test_loss = multiclass_hinge_loss(X_train, y_train, W_test, L)
print(f"Hinge loss with random W: {test_loss:.4f}")
print(f"(Expected ~1.0 since random weights give ~0 score differences, so max margin ≈ 1.0)")

Hinge loss with random W: 1.1166
(Expected ~1.0 since random weights give ~0 score differences, so max margin ≈ 1.0)


## 2. (ℓ₂)∞SVM Objective (Theorem 3.1, Eq. 13)

The paper's **Theorem 3.1** (Eq. 13) establishes that the robust SVM formulation RSVM2(σ) is equivalent to:

$$(\ell_2)_\infty\text{SVM}(\sigma): \quad \min_W \frac{1}{n} \sum_i \ell(x_i, y_i; W) + \sigma \cdot \max_y \|w_y\|_2$$

The regularizer is the **ℓ₂,∞ norm** of $W$: the maximum ℓ₂ norm among the class-specific weight vectors. This is the key contribution — it arises naturally from adversarial robustness with ℓ₂-bounded perturbations.

In [4]:
def l2_inf_svm_objective(X, y, W, sigma, L):
    """
    Eq. 13 / Theorem 3.1: (ℓ₂)∞SVM objective.
    
    (1/n) Σᵢ ℓ(xᵢ, yᵢ; W) + σ · max_y ‖w_y‖₂
    
    Parameters:
        X: (n, d) feature matrix
        y: (n,) label vector
        W: (L, d) weight matrix
        sigma: regularization parameter σ
        L: number of classes
    
    Returns:
        Scalar objective value
    """
    hinge = multiclass_hinge_loss(X, y, W, L)
    norms = np.linalg.norm(W, axis=1)  # ‖w_y‖₂ for each class y
    reg = sigma * np.max(norms)         # ℓ₂,∞ norm = max of per-class norms
    return hinge + reg

test_obj = l2_inf_svm_objective(X_train, y_train, W_test, 1.0, L)
print(f"(ℓ₂)∞SVM objective with random W, σ=1.0: {test_obj:.4f}")

(ℓ₂)∞SVM objective with random W, σ=1.0: 1.2028


## 3. Standard Frobenius SVM Objective (Eq. 2)

For comparison, the standard multiclass SVM of **Crammer & Singer** (Eq. 2 in the paper) uses Frobenius norm regularization:

$$\text{Frobenius SVM}(C): \quad \min_W \frac{1}{n} \sum_i \ell(x_i, y_i; W) + \frac{C}{2} \sum_y \|w_y\|_2^2$$

The Frobenius norm penalizes all weight vectors equally, while the ℓ₂,∞ norm only penalizes the largest one.

In [5]:
def frobenius_svm_objective(X, y, W, C, L):
    """
    Eq. 2: Standard Frobenius-regularized multiclass SVM.
    
    (1/n) Σᵢ ℓ(xᵢ, yᵢ; W) + (C/2) Σ_y ‖w_y‖²₂
    
    Parameters:
        X: (n, d) feature matrix
        y: (n,) label vector
        W: (L, d) weight matrix
        C: regularization parameter
        L: number of classes
    
    Returns:
        Scalar objective value
    """
    hinge = multiclass_hinge_loss(X, y, W, L)
    reg = (C / 2) * np.sum(W ** 2)  # (C/2) * ‖W‖²_F
    return hinge + reg

test_obj_frob = frobenius_svm_objective(X_train, y_train, W_test, 1.0, L)
print(f"Frobenius SVM objective with random W, C=1.0: {test_obj_frob:.4f}")

Frobenius SVM objective with random W, C=1.0: 1.1473


## 4. Subgradient of (ℓ₂)∞SVM Objective

To optimize the (ℓ₂)∞SVM, we use subgradient descent. The objective is convex but non-smooth (due to both the hinge loss and the max-norm regularizer), so we compute subgradients:

**Hinge loss subgradient**: For each sample $i$, find the most-violating class $\bar{y}^* = \arg\max_{\bar{y}} [w_{\bar{y}} \cdot x_i - w_{y_i} \cdot x_i + 1]$. If the margin is positive (active hinge), the subgradient updates $\partial W_{\bar{y}^*} \mathrel{+}= x_i/n$ and $\partial W_{y_i} \mathrel{-}= x_i/n$.

**ℓ₂,∞ regularizer subgradient**: Find $y^* = \arg\max_y \|w_y\|_2$. The subgradient is $\sigma \cdot w_{y^*} / \|w_{y^*}\|_2$ for row $y^*$.

In [6]:
def l2_inf_svm_subgradient(X, y, W, sigma, L):
    """
    Subgradient of the (ℓ₂)∞SVM objective w.r.t. W.
    
    Combines:
    - Hinge loss subgradient: for each sample, find ȳ* = argmax of margins.
      If margin is active (>0): dW[ȳ*] += xᵢ/n, dW[yᵢ] -= xᵢ/n
    - ℓ₂,∞ regularizer subgradient: find y* = argmax_y ‖w_y‖₂.
      dW[y*] += σ · w_{y*}/‖w_{y*}‖₂
    
    Parameters:
        X: (n, d) feature matrix
        y: (n,) label vector
        W: (L, d) weight matrix
        sigma: regularization parameter
        L: number of classes
    
    Returns:
        grad_W: (L, d) subgradient matrix
    """
    n, d = X.shape
    grad_W = np.zeros_like(W)
    
    # --- Hinge loss subgradient ---
    scores = X @ W.T  # (n, L)
    margins = scores - scores[np.arange(n), y].reshape(-1, 1) + 1.0
    margins[np.arange(n), y] = -np.inf  # exclude true class from argmax
    y_bar = np.argmax(margins, axis=1)   # most-violating class per sample
    
    # Only contribute when the hinge is active (margin > 0)
    active_margins = margins[np.arange(n), y_bar]
    active = active_margins > 0
    
    # Vectorized accumulation for hinge subgradient
    for i in range(n):
        if active[i]:
            grad_W[y_bar[i]] += X[i] / n   # misclassified class gets +xᵢ
            grad_W[y[i]] -= X[i] / n        # true class gets -xᵢ
    
    # --- ℓ₂,∞ regularizer subgradient ---
    norms = np.linalg.norm(W, axis=1)  # ‖w_y‖₂ for each class
    y_star = np.argmax(norms)           # class with largest weight norm
    if norms[y_star] > 1e-10:
        grad_W[y_star] += sigma * W[y_star] / norms[y_star]
    
    return grad_W

# Quick test
grad_test = l2_inf_svm_subgradient(X_train, y_train, W_test, 1.0, L)
print(f"Subgradient shape: {grad_test.shape}")
print(f"Subgradient norm: {np.linalg.norm(grad_test):.4f}")

Subgradient shape: (10, 64)
Subgradient norm: 2.1897


## 5. Subgradient of Frobenius SVM Objective

The Frobenius SVM has a smoother regularizer. The subgradient of the hinge loss is the same as above, and the gradient of the Frobenius regularizer $(C/2)\|W\|_F^2$ is simply $C \cdot W$.

In [7]:
def frobenius_svm_subgradient(X, y, W, C, L):
    """
    Subgradient of the Frobenius-regularized multiclass SVM objective.
    
    Same hinge subgradient as (ℓ₂)∞SVM, but the regularizer gradient is:
    dW += C * W  (gradient of (C/2)‖W‖²_F)
    
    Parameters:
        X: (n, d) feature matrix
        y: (n,) label vector
        W: (L, d) weight matrix
        C: regularization parameter
        L: number of classes
    
    Returns:
        grad_W: (L, d) subgradient matrix
    """
    n, d = X.shape
    grad_W = np.zeros_like(W)
    
    # --- Hinge loss subgradient (identical to (ℓ₂)∞SVM) ---
    scores = X @ W.T
    margins = scores - scores[np.arange(n), y].reshape(-1, 1) + 1.0
    margins[np.arange(n), y] = -np.inf
    y_bar = np.argmax(margins, axis=1)
    active_margins = margins[np.arange(n), y_bar]
    active = active_margins > 0
    
    for i in range(n):
        if active[i]:
            grad_W[y_bar[i]] += X[i] / n
            grad_W[y[i]] -= X[i] / n
    
    # --- Frobenius regularizer gradient ---
    grad_W += C * W
    
    return grad_W

# Quick test
grad_frob_test = frobenius_svm_subgradient(X_train, y_train, W_test, 1.0, L)
print(f"Frobenius subgradient shape: {grad_frob_test.shape}")
print(f"Frobenius subgradient norm: {np.linalg.norm(grad_frob_test):.4f}")

Frobenius subgradient shape: (10, 64)
Frobenius subgradient norm: 1.8813


## 6. σ Heuristic from the Paper (Section 5)

**Section 5** of the paper describes a heuristic for selecting σ based on the average nearest-neighbor distance:

$$\sigma = \frac{1}{n\sqrt{d}} \sum_{i=1}^{n} \|x_i - \text{NN}(x_i)\|_2$$

This sets the adversary's perturbation budget to be proportional to the typical distance between nearby points, scaled by the dimensionality.

In [8]:
def compute_sigma_heuristic(X):
    """
    Section 5 of the paper: σ = (1/(n√d)) Σᵢ ‖xᵢ − NN(xᵢ)‖₂
    
    Computes the heuristic value of σ based on average nearest-neighbor
    distances in the training set.
    
    Parameters:
        X: (n, d) feature matrix
    
    Returns:
        sigma: scalar heuristic value
    """
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    distances, _ = nn.kneighbors(X)
    nn_distances = distances[:, 1]  # distance to nearest neighbor (not self)
    n, d = X.shape
    sigma = np.mean(nn_distances) / np.sqrt(d)
    return sigma

sigma = compute_sigma_heuristic(X_train)
print(f"Computed σ (heuristic from Section 5): {sigma:.6f}")
print(f"Average NN distance: {np.mean(NearestNeighbors(n_neighbors=2).fit(X_train).kneighbors(X_train)[0][:, 1]):.4f}")
print(f"√d = {np.sqrt(d):.4f}")

Computed σ (heuristic from Section 5): 0.471080
Average NN distance: 3.7686
√d = 8.0000


## 7. Training Loop: Subgradient Descent

We optimize both objectives using **projected subgradient descent** with a diminishing step size $\eta_t = \eta_0 / \sqrt{t+1}$, which is a standard choice for convex non-smooth optimization and guarantees convergence of the running best objective value.

In [9]:
def train_l2_inf_svm(X, y, L, sigma, n_iters=2000, lr=0.1, eval_every=50, seed=42):
    """
    Train (ℓ₂)∞SVM using subgradient descent.
    
    Uses diminishing step size: η_t = lr / √(t+1)
    
    Parameters:
        X: (n, d) training features
        y: (n,) training labels
        L: number of classes
        sigma: regularization parameter
        n_iters: number of iterations
        lr: base learning rate
        eval_every: evaluate objective every N iterations
        seed: random seed for initialization
    
    Returns:
        W_best: (L, d) weight matrix with best objective
        losses: list of (iteration, objective) tuples
    """
    np.random.seed(seed)
    n, d = X.shape
    W = np.random.randn(L, d) * 0.01
    losses = []
    best_obj = np.inf
    W_best = W.copy()
    
    for t in range(n_iters):
        step = lr / np.sqrt(t + 1)  # diminishing step size
        grad = l2_inf_svm_subgradient(X, y, W, sigma, L)
        W = W - step * grad
        
        if t % eval_every == 0:
            obj = l2_inf_svm_objective(X, y, W, sigma, L)
            losses.append((t, obj))
            if obj < best_obj:
                best_obj = obj
                W_best = W.copy()
            if t % 500 == 0:
                print(f"  Iter {t:4d}: objective = {obj:.4f}")
    
    return W_best, losses


def train_frobenius_svm(X, y, L, C, n_iters=2000, lr=0.1, eval_every=50, seed=42):
    """
    Train Frobenius-regularized multiclass SVM using subgradient descent.
    
    Parameters:
        X: (n, d) training features
        y: (n,) training labels
        L: number of classes
        C: regularization parameter
        n_iters: number of iterations
        lr: base learning rate
        eval_every: evaluate objective every N iterations
        seed: random seed for initialization
    
    Returns:
        W_best: (L, d) weight matrix with best objective
        losses: list of (iteration, objective) tuples
    """
    np.random.seed(seed)
    n, d = X.shape
    W = np.random.randn(L, d) * 0.01
    losses = []
    best_obj = np.inf
    W_best = W.copy()
    
    for t in range(n_iters):
        step = lr / np.sqrt(t + 1)
        grad = frobenius_svm_subgradient(X, y, W, C, L)
        W = W - step * grad
        
        if t % eval_every == 0:
            obj = frobenius_svm_objective(X, y, W, C, L)
            losses.append((t, obj))
            if obj < best_obj:
                best_obj = obj
                W_best = W.copy()
            if t % 500 == 0:
                print(f"  Iter {t:4d}: objective = {obj:.4f}")
    
    return W_best, losses

print("Training functions defined.")

Training functions defined.


## 8. Prediction Function

In [10]:
def predict(X, W):
    """
    Predict class labels as argmax of scores.
    ŷ = argmax_y (w_y · x)
    """
    return np.argmax(X @ W.T, axis=1)

print("Prediction function defined.")

Prediction function defined.


## 9. Train (ℓ₂)∞SVM

We now train the (ℓ₂)∞SVM using the σ computed from the nearest-neighbor heuristic (Section 5 of the paper). This is the core model proposed by the paper.

In [11]:
print(f"Training (ℓ₂)∞SVM with σ = {sigma:.6f}")
print(f"  lr={LR}, n_iters={N_ITERS}")
print("="*50)

W_linf, losses_linf = train_l2_inf_svm(
    X_train, y_train, L=L, sigma=sigma,
    n_iters=N_ITERS, lr=LR, eval_every=EVAL_EVERY, seed=RANDOM_SEED
)

# Evaluate
y_pred_linf_train = predict(X_train, W_linf)
y_pred_linf_test = predict(X_test, W_linf)
acc_linf_train = np.mean(y_pred_linf_train == y_train)
acc_linf_test = np.mean(y_pred_linf_test == y_test)

print("="*50)
print(f"(ℓ₂)∞SVM Results:")
print(f"  Train accuracy: {acc_linf_train:.4f} ({acc_linf_train*100:.1f}%)")
print(f"  Test accuracy:  {acc_linf_test:.4f} ({acc_linf_test*100:.1f}%)")
print(f"  Test error rate: {(1-acc_linf_test)*100:.1f}%")

Training (ℓ₂)∞SVM with σ = 0.471080
  lr=0.1, n_iters=2000
  Iter    0: objective = 0.9540


  Iter  500: objective = 0.3904


  Iter 1000: objective = 0.3821


  Iter 1500: objective = 0.3787


(ℓ₂)∞SVM Results:
  Train accuracy: 0.9638 (96.4%)
  Test accuracy:  0.9444 (94.4%)
  Test error rate: 5.6%


## 10. Train Frobenius SVM (Baseline)

For comparison, we train the standard Frobenius-regularized multiclass SVM (Eq. 2). We set $C = 1/(2\sigma)$ as an approximate equivalence: in the binary case, the paper notes that $\sigma \sim 1/(2C)$ (Section 3), so this provides a rough correspondence between the two regularization strengths.

In [12]:
# Compute equivalent C for Frobenius SVM
C_equiv = 1.0 / (2 * sigma)
print(f"Training Frobenius SVM with C = 1/(2σ) = {C_equiv:.4f}")
print(f"  lr={LR}, n_iters={N_ITERS}")
print("="*50)

W_frob, losses_frob = train_frobenius_svm(
    X_train, y_train, L=L, C=C_equiv,
    n_iters=N_ITERS, lr=LR, eval_every=EVAL_EVERY, seed=RANDOM_SEED
)

# Evaluate
y_pred_frob_train = predict(X_train, W_frob)
y_pred_frob_test = predict(X_test, W_frob)
acc_frob_train = np.mean(y_pred_frob_train == y_train)
acc_frob_test = np.mean(y_pred_frob_test == y_test)

print("="*50)
print(f"Frobenius SVM Results:")
print(f"  Train accuracy: {acc_frob_train:.4f} ({acc_frob_train*100:.1f}%)")
print(f"  Test accuracy:  {acc_frob_test:.4f} ({acc_frob_test*100:.1f}%)")
print(f"  Test error rate: {(1-acc_frob_test)*100:.1f}%")

Training Frobenius SVM with C = 1/(2σ) = 1.0614
  lr=0.1, n_iters=2000
  Iter    0: objective = 0.9416


  Iter  500: objective = 0.6756


  Iter 1000: objective = 0.6756


  Iter 1500: objective = 0.6756


Frobenius SVM Results:
  Train accuracy: 0.9221 (92.2%)
  Test accuracy:  0.9083 (90.8%)
  Test error rate: 9.2%


## 11. sklearn LinearSVC Baseline

As a reference, we also train sklearn's `LinearSVC` with the Crammer-Singer formulation (`multi_class='crammer_singer'`) and the same C value. This serves as a well-optimized baseline to contextualize our subgradient-based implementations.

In [13]:
# sklearn baseline: LinearSVC with Crammer-Singer
svm_sklearn = LinearSVC(
    C=C_equiv, max_iter=5000, random_state=RANDOM_SEED,
    multi_class='crammer_singer'
)
svm_sklearn.fit(X_train, y_train)

acc_sklearn_train = svm_sklearn.score(X_train, y_train)
acc_sklearn_test = svm_sklearn.score(X_test, y_test)

print(f"sklearn LinearSVC (Crammer-Singer, C={C_equiv:.4f}) Results:")
print(f"  Train accuracy: {acc_sklearn_train:.4f} ({acc_sklearn_train*100:.1f}%)")
print(f"  Test accuracy:  {acc_sklearn_test:.4f} ({acc_sklearn_test*100:.1f}%)")
print(f"  Test error rate: {(1-acc_sklearn_test)*100:.1f}%")

sklearn LinearSVC (Crammer-Singer, C=1.0614) Results:
  Train accuracy: 1.0000 (100.0%)
  Test accuracy:  0.9444 (94.4%)
  Test error rate: 5.6%


## 12. Summary Comparison

Below we compare all three methods. The paper's key claim is that ℓ₂,∞ regularization can outperform Frobenius regularization for multiclass problems, particularly when class weight norms are imbalanced.

In [14]:
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"{'Method':<35} {'Train Acc':>10} {'Test Acc':>10} {'Test Err':>10}")
print("-"*65)
print(f"{'(ℓ₂)∞SVM (ours, Theorem 3.1)':<35} {acc_linf_train*100:>9.1f}% {acc_linf_test*100:>9.1f}% {(1-acc_linf_test)*100:>9.1f}%")
print(f"{'Frobenius SVM (ours, Eq. 2)':<35} {acc_frob_train*100:>9.1f}% {acc_frob_test*100:>9.1f}% {(1-acc_frob_test)*100:>9.1f}%")
print(f"{'sklearn LinearSVC (C-S)':<35} {acc_sklearn_train*100:>9.1f}% {acc_sklearn_test*100:>9.1f}% {(1-acc_sklearn_test)*100:>9.1f}%")
print("="*65)
print(f"\nσ (heuristic) = {sigma:.6f}")
print(f"C (equivalent) = {C_equiv:.4f}")

# Save results for Task 2.3
results = {
    'acc_linf_train': acc_linf_train,
    'acc_linf_test': acc_linf_test,
    'acc_frob_train': acc_frob_train,
    'acc_frob_test': acc_frob_test,
    'acc_sklearn_train': acc_sklearn_train,
    'acc_sklearn_test': acc_sklearn_test,
    'sigma': sigma,
    'C_equiv': C_equiv,
}
np.save('results/task_2_2_results.npy', results)
np.save('results/task_2_2_W_linf.npy', W_linf)
np.save('results/task_2_2_W_frob.npy', W_frob)
np.save('results/task_2_2_losses_linf.npy', np.array(losses_linf))
np.save('results/task_2_2_losses_frob.npy', np.array(losses_frob))
np.save('results/task_2_2_y_pred_linf_test.npy', y_pred_linf_test)
np.save('results/task_2_2_y_test.npy', y_test)
print("\nResults saved to results/task_2_2_*.npy")


RESULTS SUMMARY
Method                               Train Acc   Test Acc   Test Err
-----------------------------------------------------------------
(ℓ₂)∞SVM (ours, Theorem 3.1)             96.4%      94.4%       5.6%
Frobenius SVM (ours, Eq. 2)              92.2%      90.8%       9.2%
sklearn LinearSVC (C-S)                 100.0%      94.4%       5.6%

σ (heuristic) = 0.471080
C (equivalent) = 1.0614

Results saved to results/task_2_2_*.npy


## 13. Analysis of Weight Norms

A key difference between ℓ₂,∞ and Frobenius regularization is how they distribute weight norms across classes. The ℓ₂,∞ norm penalizes the *maximum* per-class norm, encouraging all class weight vectors to have similar norms. The Frobenius norm penalizes the *sum of squares* of all weight norms, allowing some classes to have larger weights than others.

In [15]:
# Compare weight norms across classes for both methods
norms_linf = np.linalg.norm(W_linf, axis=1)
norms_frob = np.linalg.norm(W_frob, axis=1)

print("Per-class weight norms (‖w_y‖₂):")
print(f"{'Class':<8} {'(ℓ₂)∞SVM':>12} {'Frobenius':>12}")
print("-"*35)
for c in range(L):
    print(f"{c:<8} {norms_linf[c]:>12.4f} {norms_frob[c]:>12.4f}")
print("-"*35)
print(f"{'Max':<8} {np.max(norms_linf):>12.4f} {np.max(norms_frob):>12.4f}")
print(f"{'Std':<8} {np.std(norms_linf):>12.4f} {np.std(norms_frob):>12.4f}")
print(f"\nAs expected from the ℓ₂,∞ regularizer, weight norms should be more")
print(f"uniform for (ℓ₂)∞SVM (lower std of norms) compared to Frobenius SVM.")

Per-class weight norms (‖w_y‖₂):
Class        (ℓ₂)∞SVM    Frobenius
-----------------------------------
0              0.4359       0.2226
1              0.4362       0.1808
2              0.4372       0.2101
3              0.4361       0.2013
4              0.4367       0.2187
5              0.4364       0.2267
6              0.4366       0.2114
7              0.4369       0.2170
8              0.4372       0.1547
9              0.4364       0.1625
-----------------------------------
Max            0.4372       0.2267
Std            0.0004       0.0243

As expected from the ℓ₂,∞ regularizer, weight norms should be more
uniform for (ℓ₂)∞SVM (lower std of norms) compared to Frobenius SVM.


The weight norm analysis above illustrates the geometric interpretation from the paper: the ℓ₂,∞ norm regularizer encourages the classifier to have balanced weight norms across classes, which corresponds to the adversarial robustness interpretation in Theorem 3.1.